# 1. Introduction

This notebook aims to fine-tune a CLIP model for detecting aligned and misaligned image-text pairs. The workflow includes:
- Web scraping for data collection.
- Dataset preparation.
- Model training and evaluation.
- Testing on user-uploaded photos.

Libraries and dependencies:
- `transformers` for CLIP model and processing.
- `torch` for training.
- `BeautifulSoup` for web scraping.
- `Pandas` for data manipulation.
- `sklearn` for evaluation metrics.

In [2]:
# Import dependencies
import os
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor

# 2. Web Scraping

This section collects product metadata and downloads images using web scraping. The data will be saved for reproducibility.

In [ ]:
# Directory setup
image_dir = "trendyol_dataset/product_images"
os.makedirs(image_dir, exist_ok=True)

# Web scraping logic
def download_product_image(product_url, product_id):
    try:
        response = requests.get(product_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Extract image URL
        script_tag = soup.find("script", type="application/ld+json")
        if not script_tag:
            return None
        data = json.loads(script_tag.string)
        image_urls = data.get("image", {}).get("contentUrl", [])
        if not image_urls:
            return None

        # Download image
        main_image_url = image_urls[0]
        image_response = requests.get(main_image_url, timeout=10)
        image_response.raise_for_status()
        image_path = os.path.join(image_dir, f"{product_id}.jpg")
        with open(image_path, "wb") as f:
            f.write(image_response.content)

        return image_path
    except Exception as e:
        print(f"Error downloading image for product {product_id}: {e}")
        return None

# Load dataset and add image paths
dataset_file = "trendyol_dataset/prepared_all_products.csv"
df = pd.read_csv(dataset_file)
df["image_path"] = None

# Scrape images
for index, row in tqdm(df.iterrows(), total=len(df), desc="Downloading Images"):
    product_url = row["product_link"]
    product_id = row["product_id"]
    if product_url and product_id:
        image_path = download_product_image(product_url, product_id)
        df.at[index, "image_path"] = image_path

# Save updated dataset
output_file = "trendyol_dataset/updated_trendyol_dataset_with_images.csv"
df.to_csv(output_file, index=False)
print(f"Dataset updated and saved to {output_file}")

# 3. Dataset Preparation

Generate aligned and misaligned pairs, then split the data into training and testing subsets.

In [3]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split

# Define paths
dataset_path = "trendyol_dataset"
aligned_path = os.path.join(dataset_path, "aligned_pairs.json")
misaligned_path = os.path.join(dataset_path, "misaligned_pairs.json")
train_path = os.path.join(dataset_path, "train_pairs.json")
test_path = os.path.join(dataset_path, "test_pairs.json")

# Load dataset
df = pd.read_csv(os.path.join(dataset_path, "updated_trendyol_dataset_with_images.csv"))
df = df[df["image_path"].notnull()]

# Check if JSON files already exist
if os.path.exists(aligned_path) and os.path.exists(misaligned_path):
    print("Loading aligned and misaligned pairs from JSON files...")
    with open(aligned_path, "r") as f:
        aligned_pairs = json.load(f)
    with open(misaligned_path, "r") as f:
        misaligned_pairs = json.load(f)
else:
    print("Generating aligned and misaligned pairs...")
    aligned_pairs = []
    misaligned_pairs = []

    for _, row in df.iterrows():
        aligned_pairs.append({
            "image_path": row["image_path"],
            "caption": row["title"],
            "label": "aligned"
        })

        unrelated_rows = df[df["categories"].apply(lambda x: not set(x).intersection(row["categories"]))]
        if not unrelated_rows.empty:
            random_row = unrelated_rows.sample(1).iloc[0]
            misaligned_pairs.append({
                "image_path": row["image_path"],
                "caption": random_row["title"],
                "label": "misaligned"
            })

    # Save aligned and misaligned pairs
    with open(aligned_path, "w") as f:
        json.dump(aligned_pairs, f, indent=4)

    with open(misaligned_path, "w") as f:
        json.dump(misaligned_pairs, f, indent=4)

# Check if train/test pairs exist
if os.path.exists(train_path) and os.path.exists(test_path):
    print("Loading train and test pairs from JSON files...")
    with open(train_path, "r") as f:
        train_pairs = json.load(f)
    with open(test_path, "r") as f:
        test_pairs = json.load(f)
else:
    print("Splitting data into train and test sets...")
    # Combine and split the pairs
    all_pairs = aligned_pairs + misaligned_pairs
    train_pairs, test_pairs = train_test_split(all_pairs, test_size=0.2, random_state=42)

    # Save train/test pairs
    with open(train_path, "w") as f:
        json.dump(train_pairs, f, indent=4)
    with open(test_path, "w") as f:
        json.dump(test_pairs, f, indent=4)

print("Data preparation completed!")

Loading aligned and misaligned pairs from JSON files...
Loading train and test pairs from JSON files...
Data preparation completed!


# 4. Define Dataset and DataLoader

Define a PyTorch `Dataset` and create `DataLoader` for batch processing.

In [4]:
from torch.utils.data import Dataset
from PIL import Image
from transformers import CLIPProcessor

class CLIPDataset(Dataset):
    def __init__(self, pairs, processor, max_length=77):
        """
        Initialize the dataset with pairs and processor.
        Args:
            pairs (list): A list of dictionaries containing image paths, captions, and labels.
            processor (CLIPProcessor): The CLIPProcessor for preprocessing text and images.
            max_length (int): Maximum token length for text inputs.
        """
        self.pairs = pairs
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        """Return the number of samples in the dataset."""
        return len(self.pairs)

    def __getitem__(self, idx):
        """
        Get a single sample from the dataset.
        Args:
            idx (int): The index of the sample.
        Returns:
            dict: Preprocessed inputs (text and image).
            int: Label (0 for aligned, 1 for misaligned).
        """
        pair = self.pairs[idx]
        image_path = pair["image_path"]
        text = pair["caption"]
        label = 0 if pair["label"] == "aligned" else 1

        # Load and preprocess the image
        image = Image.open(image_path).convert("RGB")

        # Preprocess the inputs with truncation and padding
        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            truncation=True,         # Kesme işlemi
            max_length=self.max_length,  # Maksimum token sayısı
            padding="max_length"    # Maksimum uzunluğa padding
        )

        return {k: v.squeeze(0) for k, v in inputs.items()}, label


# Define a custom collate function to handle padding
def collate_fn(batch):
    inputs, labels = zip(*batch)

    # Find the maximum length for input_ids and attention_mask
    max_len = max(x['input_ids'].shape[0] for x in inputs)

    # Pad all inputs to the maximum length
    padded_inputs = {
        key: torch.stack([
            torch.nn.functional.pad(
                x[key],
                pad=(0, max_len - x[key].shape[0]),
                mode='constant',
                value=0
            )
            if key in ['input_ids', 'attention_mask'] else x[key]
            for x in inputs
        ])
        for key in inputs[0]
    }

    # Convert labels to a tensor
    labels = torch.tensor(labels, dtype=torch.long)
    return padded_inputs, labels

# Load train and test pairs
with open("trendyol_dataset/train_pairs.json", "r") as f:
    train_pairs = json.load(f)

with open("trendyol_dataset/test_pairs.json", "r") as f:
    test_pairs = json.load(f)

# Create datasets and dataloaders
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
train_dataset = CLIPDataset(train_pairs, processor)
test_dataset = CLIPDataset(test_pairs, processor)

# Update the DataLoader to use the custom collate function
train_loader = DataLoader(train_dataset, batch_size=32, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

# 5. Model Definition

We use the pre-trained CLIP model and add a classification head for binary classification of aligned/misaligned pairs.

In [5]:
# Check if a GPU is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class CLIPClassifier(nn.Module):
    def __init__(self, clip_model):
        super(CLIPClassifier, self).__init__()
        self.clip_model = clip_model
        self.fc = nn.Linear(1, 2)  # Binary classification: aligned/misaligned

    def forward(self, inputs):
        outputs = self.clip_model(**inputs)
        image_features = outputs.image_embeds  # Shape: [batch_size, projection_dim]
        text_features = outputs.text_embeds  # Shape: [batch_size, projection_dim]

        # Compute similarity (dot product between embeddings)
        similarity = torch.einsum("ij,ij->i", image_features, text_features)  # Shape: [batch_size]
        
        # Add an extra dimension for linear layer
        similarity = similarity.unsqueeze(-1)  # Shape: [batch_size, 1]

        logits = self.fc(similarity)  # Shape: [batch_size, 2]
        return logits

# Load pre-trained CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

# Initialize classifier
classifier = CLIPClassifier(clip_model).to(device)

Using device: cpu


# 6. Training

This section fine-tunes the CLIP model on the aligned/misaligned pairs using the `train_loader`. Checkpoints are saved after each epoch.

## 6.1 Training with Cross-Entropy Loss

In [6]:
# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-5)
epochs = 3
save_every_n_steps = 100  # Save checkpoint every 100 steps

for epoch in range(epochs):
    classifier.train()
    total_loss = 0
    correct = 0
    total = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
        inputs, labels = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.to(device)

        # Ensure position_ids match input_ids' shape
        inputs['position_ids'] = torch.arange(0, inputs['input_ids'].size(1), device=device).unsqueeze(0).repeat(inputs['input_ids'].size(0), 1)

        # Forward pass
        logits = classifier(inputs)
        loss = criterion(logits, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update metrics
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # Save checkpoint every n steps
        if (step + 1) % save_every_n_steps == 0:
            checkpoint_path = "clip_checkpoint.pth"  # Overwrite the same file
            torch.save(classifier.state_dict(), checkpoint_path)

            # Print checkpoint details with loss and accuracy
            avg_loss = total_loss / (step + 1)
            avg_accuracy = correct / total
            print(f"Checkpoint saved at step {step+1} for epoch {epoch+1} - Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}")

    # End of epoch metrics
    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

    # Save checkpoint at the end of each epoch
    epoch_checkpoint_path = "clip_finetuned.pth"  # Overwrite the same file
    torch.save(classifier.state_dict(), epoch_checkpoint_path)
    print(f"Checkpoint saved for epoch {epoch+1}.")

Epoch 1 Training:   9%|▊         | 100/1152 [03:52<43:58,  2.51s/it]

Checkpoint saved at step 100 for epoch 1 - Loss: 0.6235, Accuracy: 0.7963


Epoch 1 Training:  11%|█▏        | 131/1152 [05:08<39:48,  2.34s/it]

## 6.2 Training with Contrastive Loss

In [7]:
import torch
from torch.nn.functional import cosine_similarity

# Contrastive Loss tanımı
def contrastive_loss(image_features, text_features, labels, margin=0.2):
    """
    Compute contrastive loss between image and text features.
    Args:
        image_features (torch.Tensor): Image embeddings (batch_size, embedding_dim)
        text_features (torch.Tensor): Text embeddings (batch_size, embedding_dim)
        labels (torch.Tensor): Labels (1 for aligned, 0 for misaligned)
        margin (float): Margin for negative samples.
    Returns:
        torch.Tensor: Loss value.
    """
    similarities = cosine_similarity(image_features, text_features)  # Cosine benzerliklerini hesapla
    positive_loss = labels * (1 - similarities) ** 2  # Aligned örnekler
    negative_loss = (1 - labels) * (similarities - margin).clamp(min=0) ** 2  # Misaligned örnekler
    return (positive_loss + negative_loss).mean()

from transformers import CLIPModel
from tqdm import tqdm

# CLIP modelini yükle ve eğitime aç
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
for param in clip_model.parameters():
    param.requires_grad = True  # Tüm CLIP modelini eğitime dahil et

# Fine-tuning model
class CLIPFineTuner(torch.nn.Module):
    def __init__(self, clip_model):
        super(CLIPFineTuner, self).__init__()
        self.clip_model = clip_model

    def forward(self, inputs):
        outputs = self.clip_model(**inputs)
        image_features = outputs.image_embeds  # Görsel özellikler
        text_features = outputs.text_embeds  # Metin özellikler
        return image_features, text_features

# Model ve optimizer
fine_tuner = CLIPFineTuner(clip_model).to(device)
optimizer = torch.optim.AdamW(fine_tuner.parameters(), lr=1e-6)
epochs = 3
save_every_n_steps = 100

# Eğitim döngüsü
for epoch in range(epochs):
    fine_tuner.train()
    total_loss = 0
    correct = 0
    total = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Fine-Tuning")):
        inputs, labels = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.float().to(device)  # Contrastive loss için float kullan

        # Forward pass
        image_features, text_features = fine_tuner(inputs)

        # Loss hesaplama
        loss = contrastive_loss(image_features, text_features, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update metrics
        total_loss += loss.item()
        preds = (cosine_similarity(image_features, text_features) > 0.5).float()  # Benzerlik eşik değerine göre tahmin
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # Ara checkpoint kaydı
        if (step + 1) % save_every_n_steps == 0:
            checkpoint_path = "clip_fine_tune_checkpoint.pth"
            torch.save(fine_tuner.state_dict(), checkpoint_path)
            avg_loss = total_loss / (step + 1)
            avg_accuracy = correct / total
            print(f"Checkpoint saved at step {step+1} for epoch {epoch+1} - Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}")

    # Epoch sonu metrikleri
    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

    # Epoch sonunda checkpoint kaydı
    torch.save(fine_tuner.state_dict(), f"clip_fine_tuned_epoch_{epoch+1}.pth")
    print(f"Checkpoint saved for epoch {epoch+1}.")

Epoch 1 Fine-Tuning:   9%|▊         | 100/1152 [04:12<46:02,  2.63s/it]

Checkpoint saved at step 100 for epoch 1 - Loss: 0.2143, Accuracy: 0.4816


Epoch 1 Fine-Tuning:  17%|█▋        | 200/1152 [08:04<40:30,  2.55s/it]

Checkpoint saved at step 200 for epoch 1 - Loss: 0.1887, Accuracy: 0.4925


Epoch 1 Fine-Tuning:  26%|██▌       | 300/1152 [12:03<36:14,  2.55s/it]

Checkpoint saved at step 300 for epoch 1 - Loss: 0.1796, Accuracy: 0.4944


... (62 outputs left out)

Epoch 3 Fine-Tuning:  95%|█████████▌| 1100/1152 [43:31<02:06,  2.44s/it]

Checkpoint saved at step 1100 for epoch 3 - Loss: 0.0918, Accuracy: 0.8040


Epoch 3 Fine-Tuning: 100%|██████████| 1152/1152 [45:36<00:00,  2.38s/it]


Epoch 3, Loss: 0.0916, Accuracy: 0.8046
Checkpoint saved for epoch 3.


# 7. Evaluation

Evaluate the fine-tuned model on the `test_loader` and calculate metrics like accuracy and F1-score.

In [8]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Evaluation loop
classifier.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        inputs, labels = batch
        inputs = {k: v.squeeze().to(device) for k, v in inputs.items()}
        labels = labels.to(device)

        # Forward pass
        logits = classifier(inputs)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Metrics
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=["Aligned", "Misaligned"])

print(f"Test Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Classification Report:")
print(report)

Evaluating: 100%|██████████| 288/288 [05:01<00:00,  1.05s/it]

Test Accuracy: 0.8479
F1 Score: 0.8361
Classification Report:
              precision    recall  f1-score   support

     Aligned       0.80      0.92      0.86      4608
  Misaligned       0.91      0.78      0.84      4607

    accuracy                           0.85      9215
   macro avg       0.86      0.85      0.85      9215
weighted avg       0.86      0.85      0.85      9215



In [21]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Test for contrastive evaluation
correct = 0
total = 0
all_labels = []
all_preds = []

fine_tuner.eval()  # Set the model to evaluation mode

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Contrastive Model"):
        inputs, labels = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.to(device)

        # Get image and text features
        image_features, text_features = fine_tuner(inputs)

        # Compute cosine similarity
        similarities = torch.nn.functional.cosine_similarity(image_features, text_features)

        # Apply threshold (e.g., 0.5) for predictions
        preds = (similarities > 0.5).long()

        # Collect predictions and labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute performance metrics
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
report = classification_report(all_labels, all_preds, target_names=["Aligned", "Misaligned"])

print(f"Contrastive Test Accuracy: {accuracy:.4f}")
print(f"Contrastive F1 Score: {f1:.4f}")
print("Contrastive Classification Report:")
print(report)

Evaluating Contrastive Model: 100%|██████████| 288/288 [05:01<00:00,  1.05s/it]

Contrastive Test Accuracy: 0.7901
Contrastive F1 Score: 0.7896
Contrastive Classification Report:
              precision    recall  f1-score   support

     Aligned       0.76      0.84      0.80      4608
  Misaligned       0.82      0.74      0.78      4607

    accuracy                           0.79      9215
   macro avg       0.79      0.79      0.79      9215
weighted avg       0.79      0.79      0.79      9215



# 8. Testing User Photos

This section tests the model's ability to determine whether a user-uploaded photo matches a given product.

In [27]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from PIL import Image
import os

# CLIP modelini yükleme
from transformers import CLIPModel, CLIPProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Klasör yolları
product_image_dir = "trendyol_dataset/product_images"
review_image_dir = "trendyol_dataset/review_images"

# Test veri kümesi (örnek)
# Her biri (product_image_name, review_image_name, label) içeren bir liste
test_data = [
    ("160146962.jpg", "160146962-0-1.jpg", 1),  # Uyumlu
    ("160146962.jpg", "160146962-2-0.jpg", 0),  # Uyumsuz
    ("160146962.jpg", "160146962-1-1.jpg", 1),  # Uyumlu
    ("160146962.jpg", "160146962-3-0.jpg", 0),  # Uyumsuz
]

# Eşik değeri
threshold = 0.45

# Tahminler ve gerçek etiketler
all_preds = []
all_labels = []

# Değerlendirme döngüsü
for product_name, review_name, label in tqdm(test_data, desc="Evaluating"):
    # Ürün görseli yolu
    product_path = os.path.join(product_image_dir, product_name)
    product_image = Image.open(product_path).convert("RGB").resize((224, 224))
    product_inputs = processor(images=[product_image], return_tensors="pt").to(device)

    # Kullanıcı yorum görseli yolu
    review_path = os.path.join(review_image_dir, review_name)
    user_image = Image.open(review_path).convert("RGB").resize((224, 224))
    user_inputs = processor(images=[user_image], return_tensors="pt").to(device)

    # Embedding çıkarımı
    with torch.no_grad():
        product_embedding = clip_model.get_image_features(**product_inputs)
        user_embedding = clip_model.get_image_features(**user_inputs)

    # Normalize embedding'ler
    product_embedding = F.normalize(product_embedding, dim=1).cpu().numpy()
    user_embedding = F.normalize(user_embedding, dim=1).cpu().numpy()

    # Benzerlik hesaplama
    similarity = cosine_similarity(product_embedding, user_embedding)[0][0]

    # Tahmin (threshold ile)
    pred = 1 if similarity > threshold else 0

    # Tahmini ve gerçek etiketi kaydet
    all_preds.append(pred)
    all_labels.append(label)

# Performans metrikleri
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=["Misaligned", "Aligned"])

# Sonuçları yazdırma
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("\nClassification Report:")
print(report)

Evaluating: 100%|██████████| 4/4 [00:00<00:00, 11.78it/s]

Accuracy: 0.7500
F1 Score: 0.8000
Precision: 0.6667
Recall: 1.0000

Classification Report:
              precision    recall  f1-score   support

  Misaligned       1.00      0.50      0.67         2
     Aligned       0.67      1.00      0.80         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4

